In [1]:
import zarr
import napari
import numpy as np
import scipy.ndimage as ndi
import dask.array as da
import skimage as ski
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import h5py

In [2]:
viewer = napari.Viewer()

In [35]:
fname = 'S:/micro/mg2/ak2718/20250418_DragonFly_mck/2025-04-18/ncol5_488_ncol1_546_dapi_ncol4_647_s06.ims'

In [36]:
reader = h5py.File(fname)

In [37]:
# Full resolution

tpoints = []
for tpoint in reader['DataSet']['ResolutionLevel 0'].keys():    
    channels = []
    for channel in reader['DataSet']['ResolutionLevel 0'][tpoint].keys():
        if 'Data' in reader['DataSet']['ResolutionLevel 0'][tpoint][channel]:
            data = da.from_array(reader['DataSet']['ResolutionLevel 0'][tpoint][channel]['Data'], chunks=(8,1024,1024))
            channels.append(data)
        else:
            print('No data found')
    channels = da.stack(channels)
    tpoints.append(channels)
img = da.squeeze(da.stack(tpoints, axis=0))

In [38]:
img

dask.array<getitem, shape=(5, 80, 2048, 2048), dtype=uint16, chunksize=(1, 8, 1024, 1024), chunktype=numpy.ndarray>

In [39]:
# All resolutions

resolutions = []
for resolution in reader['DataSet'].keys():
    tpoints = []
    for tpoint in reader['DataSet']['ResolutionLevel 0'].keys():    
        channels = []
        for channel in reader['DataSet']['ResolutionLevel 0'][tpoint].keys():
            if 'Data' in reader['DataSet']['ResolutionLevel 0'][tpoint][channel]:
                data = da.from_array(reader['DataSet']['ResolutionLevel 0'][tpoint][channel]['Data'], chunks=(8,1024,1024))
                channels.append(data)
            else:
                print('No data found')
        channels = da.stack(channels)
        tpoints.append(channels)
    tpoints = da.stack(tpoints, axis=0)
    tpoints = da.squeeze(tpoints)
    resolutions.append(tpoints)


In [40]:
img.shape

(5, 80, 2048, 2048)

In [41]:
img = img.compute()

In [42]:
ski.io.imsave(fname.replace('.ims', '.tif'), img)